In [1]:
!pip -q install transformers datasets torch scikit-learn evaluate accelerate wandb

In [2]:
from datasets import load_dataset
import csv
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", 
    column_names=col_names,
    quoting=csv.QUOTE_NONE
)

# Mapping des 6 classes
label_mapping = {
    'pants-fire': 0, 'false': 1, 'barely-true': 2, 
    'half-true': 3, 'mostly-true': 4, 'true': 5
}
target_names = ['Pants-Fire', 'False', 'Barely-True', 'Half-True', 'Mostly-True', 'True']

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import datasets
import wandb
import os
from transformers import BertConfig

my_secret_key = os.environ.get("WANDB")

wandb.login(key=my_secret_key)

wandb.init(project="liar-bert-fine-grained")

statement_train, y_train = raw_datasets["train"]["statement"], raw_datasets["train"]["label"]
statement_val, y_val = raw_datasets["validation"]["statement"], raw_datasets["validation"]["label"]
statement_test, y_test = raw_datasets["test"]["statement"], raw_datasets["test"]["label"]

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)


class StatementDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    max_length=self.max_len,
                    padding='max_length',
                    truncation=True,
                    return_attention_mask=True,
                    return_tensors='pt',
                )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_dataset = StatementDataset(statement_train, y_train, tokenizer)
val_dataset = StatementDataset(statement_val, y_val, tokenizer)
test_dataset = StatementDataset(statement_test, y_test, tokenizer)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 5. Chargement du modèle
# num_labels doit correspondre au nombre de classes dans vos données (ex: 2 pour binaire)
num_labels = len(set(y_train)) 

# 1. Charger la configuration par défaut
config = BertConfig.from_pretrained(model_name, num_labels=num_labels)

# 2. Augmenter le Dropout (Par défaut c'est 0.1)
# hidden_dropout_prob : Dropout sur les couches cachées (embeddings, encodeurs)
config.hidden_dropout_prob = 0.3  
# attention_probs_dropout_prob : Dropout sur les mécanismes d'attention
config.attention_probs_dropout_prob = 0.3 

# 3. Charger le modèle avec cette configuration modifiée
model = BertForSequenceClassification.from_pretrained(model_name, config=config)

# 6. Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir='./results',          # Dossier de sortie
    num_train_epochs=5,              # Nombre d'époques
    per_device_train_batch_size=16,  # Taille du batch d'entraînement
    per_device_eval_batch_size=32,   # Taille du batch d'évaluation
    warmup_steps=500,                # Steps de chauffe pour le learning rate
    weight_decay=0.01,               # Régularisation
    logging_dir='./logs',
    report_to="wandb",          # Active l'intégration Weights & Biases
    
    logging_steps=10,           # Enregistre la TRAINING loss tous les 10 batches
    
    eval_strategy="steps",      # Permet d'évaluer PENDANT l'époque (et non juste à la fin)
    eval_steps=50,              # Lance la validation tous les 50 batches (ajuster selon vitesse)
    
    save_strategy="steps",
    save_steps=500,
    
    load_best_model_at_end=True,
    metric_for_best_model="f1"  # Optionnel : choisit le meilleur modèle basé sur le F1 score plutôt que la loss
)

# 7. Création du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 8. Lancer l'entraînement
trainer.train()

wandb.finish()

# 9. Evaluation finale sur le test set
results = trainer.evaluate(test_dataset)
print("Résultats sur le test set :", results)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/onyxia/.netrc
wandb: Currently logged in as: elie-attali (elie-attali-ensae) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 411.82it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,1.847894,1.796028,0.201713,0.133320,0.130869,0.201713
100,1.782922,1.775223,0.204050,0.152662,0.140488,0.204050
150,1.777226,1.775321,0.206386,0.130676,0.135277,0.206386
200,1.769819,1.760993,0.207944,0.118914,0.217242,0.207944
250,1.720583,1.802103,0.200935,0.103308,0.243779,0.200935
300,1.742034,1.746139,0.209502,0.112184,0.175221,0.209502
350,1.755957,1.756898,0.207165,0.097982,0.105734,0.207165
400,1.750451,1.751867,0.248442,0.178043,0.151259,0.248442
450,1.721586,1.734266,0.247664,0.140815,0.099483,0.247664
500,1.755215,1.730627,0.238318,0.196742,0.171476,0.238318


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-pa

eval/accuracy,▁▁▂▂▁▂▂▆▆▅▃▆▇▅▇▅▄▅▅▇▇█▇▇▇▇█▆▇▇▇▆█▇▇▇▇█
eval/f1,▃▃▂▂▁▂▁▄▃▅▃▃▅▇▇▆▄▅▆▆▆▆▅▆▆▆█▇█▇▇▇▇▇▇▇▇▇
eval/loss,█▆▆▆█▅▅▅▄▄▄▃▄▃▃▂▅▃▂▂▂▂▁▄▂▃▂▁▁▂▁▂▂▁▂▂▂▁
eval/precision,▂▂▂▃▄▂▁▂▁▂▂▂▂▄▆▄█▅▅▄▄▅▅▅▄▅▄▄▄▄▄▄▄▄▄▄▄▄
eval/recall,▁▁▂▂▁▂▂▆▆▅▃▆▇▅▇▅▄▅▅▇▇█▇▇▇▇█▆▇▇▇▆█▇▇▇▇█
eval/runtime,▁▂▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅█▅▅▅▅▅▅▅▅▅▅▆▅▅▅▅▅▅
eval/samples_per_second,█▇▆▅▄▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▁▄▄▄▄▄▄▄▄▃▄▃▄▄▄▄▄▄
eval/steps_per_second,█▇▆▅▄▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▁▄▄▄▄▄▄▄▄▃▄▃▄▄▄▄▄▄
train/epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
+3,...


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Error: You must call wandb.init() before wandb.log()